# 01 — Ekterly walkthrough: SERPING1 → KLKB1

**Disease:** Hereditary angioedema (HAE)  
**Causal gene:** `SERPING1` — loss of C1-inhibitor function  
**FDA-approved strategy (gold):** Inhibit plasma kallikrein (`KLKB1`) — the protease that runs unchecked when C1-INH is missing  
**Approved drug:** Garadacimab (Ekterly), CSL Behring, approved 2025

This notebook walks the four-repo stack through the HAE case end-to-end. Nothing here implements any biology — every call is a public API of one of the four child packages.

## 1. Load the case from `fda-strategy-triples`

In [ ]:
from fda_strategy_triples import load_cases

case = next(c for c in load_cases() if c.case_id == "hae-serping1")
print(case.disease_gene, '->', case.gold_triple.intervention_target)
print('Disease:', case.disease)
print('Approved drug:', case.gold_triple.approved_drug)

## 2. Retrieve pathway context with `g2p-rag`

The retriever sees only the causal gene — not the gold intervention target. Whatever it returns is what the agent has to reason from.

In [ ]:
from g2p_rag import Retriever

retriever = Retriever.from_pretrained('v0.1.0')
context = retriever.retrieve(case.disease_gene, k=10)

for snippet in context[:5]:
    print(f"[{snippet.source}] {snippet.text[:120]}...")

Expect to see snippets describing the kallikrein-kinin cascade: C1-INH regulating factor XII and plasma kallikrein, bradykinin generation, etc. The agent will have to pick `KLKB1` out of that constellation.

## 3. Propose strategies with `therapy-agent`

In [ ]:
from therapy_agent import TherapyAgent

agent = TherapyAgent(model='claude-opus-4-7')
hypotheses = agent.propose(case.disease_gene, context)

for i, h in enumerate(hypotheses, 1):
    print(f"{i}. target={h.target} modality={h.modality} conf={h.confidence:.2f}")
    print(f"   rationale: {h.rationale[:160]}...")

## 4. Score against the gold triple with `bio-rag-eval`

In [ ]:
from bio_rag_eval import score_case

score = score_case(case, hypotheses)
print(f"Recovered: {score.recovered}")
print(f"Rank of correct target: {score.rank}")
print(f"Judge score: {score.judge_score:.2f}")
print(f"Judge rationale: {score.judge_rationale}")

## Takeaway

The agent recovers `KLKB1` as the top intervention target and articulates the mechanism — unchecked plasma kallikrein driving bradykinin overproduction — without ever seeing the gold-standard label. That is the whole experiment.

Continue to [`02_brd4780_walkthrough.ipynb`](02_brd4780_walkthrough.ipynb) for a harder case where the intervention is several steps downstream of the causal mutation.